# 08 Bootstrap CIs and Report Tables

Create final metric tables and bootstrap confidence intervals from score files produced by earlier notebooks. Baseline CIs are generated from notebook 06 outputs; AE CIs are generated automatically if notebook 05 score files are available.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score


In [2]:
REPO_ROOT = Path('..').resolve()
DATASET_CONFIGS = {
    'turning': {'positive_label': 'chatter'},
    'cnc_machining': {'positive_label': 'anomaly'},
}
DATASETS_TO_REPORT = tuple(DATASET_CONFIGS)
SCORE_DIR = REPO_ROOT / 'reports' / 'scores'
BASELINE_DIR = REPO_ROOT / 'reports' / 'baselines'
THRESHOLD_DIR = REPO_ROOT / 'reports' / 'thresholds'
TABLE_DIR = REPO_ROOT / 'reports' / 'tables'
FIGURE_DIR = REPO_ROOT / 'reports' / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

N_BOOTSTRAP = 1000
RANDOM_SEED = 42
AE_SCORE_COLUMNS = ['global_mse', 'global_mae', 'ver_max', 'ver_topk']


In [3]:
def evaluate_at_threshold(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'pr_auc': float(average_precision_score(y_true, scores)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }

def bootstrap_intervals(y_true: np.ndarray, scores: np.ndarray, threshold: float, n_bootstrap=N_BOOTSTRAP, seed=RANDOM_SEED) -> dict:
    rng = np.random.default_rng(seed)
    n = len(y_true)
    rows = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        sampled_y = y_true[idx]
        sampled_scores = scores[idx]
        if len(np.unique(sampled_y)) < 2:
            continue
        rows.append(evaluate_at_threshold(sampled_y, sampled_scores, threshold))
    boot = pd.DataFrame(rows)
    intervals = {}
    for metric in ['pr_auc', 'f1', 'precision', 'recall']:
        intervals[f'{metric}_ci_low'] = float(boot[metric].quantile(0.025))
        intervals[f'{metric}_ci_high'] = float(boot[metric].quantile(0.975))
    intervals['bootstrap_n_effective'] = int(len(boot))
    return intervals


In [4]:
rows = []

for dataset_name in DATASETS_TO_REPORT:
    baseline_scores_path = BASELINE_DIR / f'baseline_scores_{dataset_name}.csv'
    baseline_thresholds_path = THRESHOLD_DIR / f'baseline_thresholds_{dataset_name}.json'
    if not baseline_scores_path.exists() or not baseline_thresholds_path.exists():
        print(f'Baseline score files not found for {dataset_name}; skipping.')
        continue

    baseline_scores = pd.read_csv(baseline_scores_path)
    with baseline_thresholds_path.open() as f:
        baseline_thresholds = json.load(f)
    for method, group in baseline_scores[baseline_scores['split'] == 'test'].groupby('method'):
        threshold = float(baseline_thresholds[method]['threshold'])
        y_true = group['target'].to_numpy()
        scores = group['score_value'].to_numpy()
        metrics = evaluate_at_threshold(y_true, scores, threshold)
        intervals = bootstrap_intervals(y_true, scores, threshold)
        rows.append({'dataset': dataset_name, 'method': method, 'score': 'anomaly_score', 'threshold': threshold, **metrics, **intervals})

baseline_ci = pd.DataFrame(rows)
if not baseline_ci.empty:
    combined_path = TABLE_DIR / 'metrics_baselines_all_datasets_with_ci.csv'
    baseline_ci.to_csv(combined_path, index=False)
    print(f'Wrote {combined_path}')
    for dataset_name, group in baseline_ci.groupby('dataset'):
        output_path = TABLE_DIR / f'metrics_{dataset_name}_baselines_with_ci.csv'
        group.to_csv(output_path, index=False)
        print(f'Wrote {output_path}')
baseline_ci


Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/tables/metrics_baselines_all_datasets_with_ci.csv
Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/tables/metrics_cnc_machining_baselines_with_ci.csv
Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/tables/metrics_turning_baselines_with_ci.csv


,dataset,method,score,threshold,pr_auc,f1,precision,recall,tn,fp,...,tp,pr_auc_ci_low,pr_auc_ci_high,f1_ci_low,f1_ci_high,precision_ci_low,precision_ci_high,recall_ci_low,recall_ci_high,bootstrap_n_effective
0,turning,isolation_forest_features,anomaly_score,-0.039756,0.990143,0.928571,0.896552,0.962963,45,3,...,26,0.966092,1.000000,0.842105,0.985075,0.750000,1.000000,0.869493,1.000000,1000
1,turning,one_class_svm_features,anomaly_score,0.253464,0.959115,0.905660,0.923077,0.888889,46,2,...,24,0.897136,0.993973,0.800000,0.978723,0.807661,1.000000,0.740741,1.000000,1000
2,turning,pca_image_reconstruction,anomaly_score,0.001279,0.961881,0.947368,0.900000,1.000000,45,3,...,27,0.891160,1.000000,0.861946,1.000000,0.757386,1.000000,1.000000,1.000000,1000
3,cnc_machining,isolation_forest_features,anomaly_score,-0.131895,0.149312,0.249124,0.142953,0.968218,173,5114,...,853,0.138234,0.161955,0.235431,0.262073,0.134003,0.151532,0.957106,0.979095,1000
4,cnc_machining,one_class_svm_features,anomaly_score,-32.893099,0.139689,0.250213,0.143043,0.997730,21,5266,...,879,0.128617,0.152702,0.237287,0.262805,0.134615,0.151379,0.994205,1.000000,1000
5,cnc_machining,pca_image_reconstruction,anomaly_score,0.000015,0.134950,0.256681,0.147411,0.992054,232,5055,...,874,0.125066,0.146150,0.243428,0.269461,0.138767,0.155908,0.985568,0.997701,1000


In [5]:
ae_rows = []

for dataset_name in DATASETS_TO_REPORT:
    ae_score_path = SCORE_DIR / f'ae_scores_{dataset_name}.csv'
    ae_threshold_path = THRESHOLD_DIR / f'ae_thresholds_{dataset_name}.json'
    if not ae_score_path.exists() or not ae_threshold_path.exists():
        print(f'AE score files not found for {dataset_name}; skipping.')
        continue

    scores = pd.read_csv(ae_score_path)
    with ae_threshold_path.open() as f:
        thresholds = json.load(f)
    test_scores = scores[scores['split'] == 'test'].copy()
    if 'target' not in test_scores.columns:
        test_scores['target'] = (test_scores['label'] == DATASET_CONFIGS[dataset_name]['positive_label']).astype(int)
    y_true = test_scores['target'].to_numpy()
    for score_name in AE_SCORE_COLUMNS:
        threshold = float(thresholds[score_name]['threshold'])
        score_values = test_scores[score_name].to_numpy()
        metrics = evaluate_at_threshold(y_true, score_values, threshold)
        intervals = bootstrap_intervals(y_true, score_values, threshold)
        ae_rows.append({'dataset': dataset_name, 'method': 'cnn_ae', 'score': score_name, 'threshold': threshold, **metrics, **intervals})

ae_ci = pd.DataFrame(ae_rows)
if not ae_ci.empty:
    combined_path = TABLE_DIR / 'metrics_ae_all_datasets_with_ci.csv'
    ae_ci.to_csv(combined_path, index=False)
    print(f'Wrote {combined_path}')
    for dataset_name, group in ae_ci.groupby('dataset'):
        output_path = TABLE_DIR / f'metrics_{dataset_name}_ae_with_ci.csv'
        group.to_csv(output_path, index=False)
        print(f'Wrote {output_path}')
ae_ci


Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/tables/metrics_ae_all_datasets_with_ci.csv
Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/tables/metrics_cnc_machining_ae_with_ci.csv
Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/tables/metrics_turning_ae_with_ci.csv


,dataset,method,score,threshold,pr_auc,f1,precision,recall,tn,fp,...,tp,pr_auc_ci_low,pr_auc_ci_high,f1_ci_low,f1_ci_high,precision_ci_low,precision_ci_high,recall_ci_low,recall_ci_high,bootstrap_n_effective
0,turning,cnn_ae,global_mse,0.002134,0.979938,0.931034,0.870968,1.000000,44,4,...,27,0.934686,1.000000,0.836364,0.985075,0.718750,0.970588,1.000000,1.00000,1000
1,turning,cnn_ae,global_mae,0.029933,0.993651,0.928571,0.896552,0.962963,45,3,...,26,0.975624,1.000000,0.842105,0.985075,0.750000,1.000000,0.869493,1.00000,1000
2,turning,cnn_ae,ver_max,0.003497,0.965831,0.915254,0.843750,1.000000,43,5,...,27,0.903278,0.999131,0.816189,0.978745,0.689459,0.958375,1.000000,1.00000,1000
3,turning,cnn_ae,ver_topk,0.002744,0.968217,0.915254,0.843750,1.000000,43,5,...,27,0.909413,1.000000,0.816327,0.981149,0.689655,0.962996,1.000000,1.00000,1000
4,cnc_machining,cnn_ae,global_mse,0.000013,0.068937,0.159481,0.086681,0.995842,97,5047,...,479,0.062279,0.077223,0.146018,0.173600,0.078773,0.095070,0.989244,1.00000,1000
5,cnc_machining,cnn_ae,global_mae,0.000348,0.069065,0.160630,0.087361,0.995842,140,5004,...,479,0.062428,0.077112,0.147038,0.174670,0.079382,0.095788,0.989244,1.00000,1000
6,cnc_machining,cnn_ae,ver_max,0.000028,0.074791,0.158252,0.085971,0.993763,62,5082,...,478,0.067334,0.084074,0.144665,0.172040,0.078028,0.094151,0.985712,1.00000,1000
7,cnc_machining,cnn_ae,ver_topk,0.000032,0.071809,0.160270,0.087229,0.985447,184,4960,...,474,0.064612,0.080511,0.146753,0.174409,0.079275,0.095591,0.973736,0.99586,1000


## Combined Comparison

Combine baseline and CNN autoencoder intervals into one comparison table when both families are available.


In [6]:
comparison_frames = []
if not baseline_ci.empty:
    comparison_frames.append(baseline_ci.assign(model_family='baseline'))
if not ae_ci.empty:
    comparison_frames.append(ae_ci.assign(model_family='cnn_ae'))

comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not comparison.empty:
    comparison_path = TABLE_DIR / 'metrics_baseline_vs_cnn_ae_all_datasets_with_ci.csv'
    comparison.to_csv(comparison_path, index=False)
    print(f'Wrote {comparison_path}')
comparison


Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/tables/metrics_baseline_vs_cnn_ae_all_datasets_with_ci.csv


,dataset,method,score,threshold,pr_auc,f1,precision,recall,tn,fp,...,pr_auc_ci_low,pr_auc_ci_high,f1_ci_low,f1_ci_high,precision_ci_low,precision_ci_high,recall_ci_low,recall_ci_high,bootstrap_n_effective,model_family
0,turning,isolation_forest_features,anomaly_score,-0.039756,0.990143,0.928571,0.896552,0.962963,45,3,...,0.966092,1.000000,0.842105,0.985075,0.750000,1.000000,0.869493,1.000000,1000,baseline
1,turning,one_class_svm_features,anomaly_score,0.253464,0.959115,0.905660,0.923077,0.888889,46,2,...,0.897136,0.993973,0.800000,0.978723,0.807661,1.000000,0.740741,1.000000,1000,baseline
2,turning,pca_image_reconstruction,anomaly_score,0.001279,0.961881,0.947368,0.900000,1.000000,45,3,...,0.891160,1.000000,0.861946,1.000000,0.757386,1.000000,1.000000,1.000000,1000,baseline
3,cnc_machining,isolation_forest_features,anomaly_score,-0.131895,0.149312,0.249124,0.142953,0.968218,173,5114,...,0.138234,0.161955,0.235431,0.262073,0.134003,0.151532,0.957106,0.979095,1000,baseline
4,cnc_machining,one_class_svm_features,anomaly_score,-32.893099,0.139689,0.250213,0.143043,0.997730,21,5266,...,0.128617,0.152702,0.237287,0.262805,0.134615,0.151379,0.994205,1.000000,1000,baseline
5,cnc_machining,pca_image_reconstruction,anomaly_score,0.000015,0.134950,0.256681,0.147411,0.992054,232,5055,...,0.125066,0.146150,0.243428,0.269461,0.138767,0.155908,0.985568,0.997701,1000,baseline
6,turning,cnn_ae,global_mse,0.002134,0.979938,0.931034,0.870968,1.000000,44,4,...,0.934686,1.000000,0.836364,0.985075,0.718750,0.970588,1.000000,1.000000,1000,cnn_ae
7,turning,cnn_ae,global_mae,0.029933,0.993651,0.928571,0.896552,0.962963,45,3,...,0.975624,1.000000,0.842105,0.985075,0.750000,1.000000,0.869493,1.000000,1000,cnn_ae
8,turning,cnn_ae,ver_max,0.003497,0.965831,0.915254,0.843750,1.000000,43,5,...,0.903278,0.999131,0.816189,0.978745,0.689459,0.958375,1.000000,1.000000,1000,cnn_ae
9,turning,cnn_ae,ver_topk,0.002744,0.968217,0.915254,0.843750,1.000000,43,5,...,0.909413,1.000000,0.816327,0.981149,0.689655,0.962996,1.000000,1.000000,1000,cnn_ae
